(rasterising-lines-section)=
# Drawing lines

One of the fundamental tasks in computer graphics is the rendering of a straight line on a display. Consider the diagram in {numref}`line-raster-figure` showing the rasterisation of the straight line joining the two points with pixel co-ordinates $(2, 2)$ and $(12, 6)$. The pixels that are closest to the line are shaded to create a rasterised approximation of the idealise image. This is a simple task for a human since we are able to view the idealised image and determine which pixels need to be shaded. Algorithms are required to determine which pixels to illuminate in order to rasterise straight lines

```{figure} /images/rasterising_line.svg
:name: line-raster-figure
:width: 400px

Rasterising a straight line
```

## Bresenham's algorithm

```{figure} https://www.ithistory.org/sites/default/files/honor-roll/Jack%20E.%20Bresenham.jpg
---
width: 200px
alt: Jack Elton Bresenham
figclass: margin
---
[Jack Elton Bresenham](https://en.wikipedia.org/wiki/Jack_Elton_Bresenham)
```

**Bresenham's algorithm** developed by American computer scientist Jack {cite:t}`bresenham1965algorithm` is a line drawing algorithm that uses integer only arithmetic and offers a vast improvement over other methods in terms of computational efficiency. Consider the <a href="https://en.wikipedia.org/wiki/Linear_equation" target="_blank">Cartesian equation of a straight line</a> joining two points with co-ordinates $(x_0, y_0)$ and $(x_1, y_1)$

```{math}
:label: cartesian-equation-of-a-straight-line

y = \frac{y_1 - y_0}{x_1 - x_0} x + c,
```

where $(x,y)$ are the co-ordinates of a point on the line and $c$ is the point of interception between the line and the $y$-axis. Let $\Delta y = y_1 - y_0$ and $\Delta x = x_1 - x_0$ into equation {eq}`cartesian-equation-of-a-straight-line` and rearranging gives

\begin{align*}
    y &= \frac{\Delta y}{\Delta x} x + c \\
    y \Delta x &= x \Delta y + c \Delta x \\
    0 &= x \Delta y - y \Delta x + c \Delta x
\end{align*}

Let $f(x,y) = x \Delta y - y \Delta x + c \Delta x$ then the sign of $f(x,y)$ tells us whether the pixel at $(x,y)$ is above or below the line such that

\begin{align*}
    f(x,y) 
    \begin{cases} 
        \geq 0, & (x,y) \text{ is on or above the line,} \\
        < 0, & (x,y) \text{ is below the line.}
    \end{cases}
\end{align*}

To simplify the derivation we are going to assume that the two end points of the line at $(x_0,y_0)$ and $(x_1,y_1)$ are such that $x_0 < x_1$ and $y_0 < y_1$ so the line is drawn from left-to-right and top-to-bottom (remember that the $y$-axis increases as we move down the raster). If the pixel at $(x_i,y_i)$ has been plotted we move one pixel to the right and we a choice between two candidate pixels at $(x_i+1,y_i)$ and $(x_i+1,y_i+1)$. To decide between these two we calculate the value of $f(x,y)$ at the midpoint between these two pixels and use its sign. Let $d = f(x_i+1,y_i+\frac{1}{2})$ be a **decision variable** then

$$ d = (x_i + 1) \Delta y - (y_i + \tfrac{1}{2}) \Delta x + c \Delta x. $$

```{figure} /images/bresenhams_algorithm.svg
:width: 400px
:name: bresenham-figure

The sign of the decision variable $d$ determines which pixel is closest to the line.
```

So the sign of $d$ will tell use which of the candidate nodes to choose ({numref}`bresenham-figure`). When $d \leq 0$ the pixel at $(x_i+1,y_i)$ is closer to the line and if $d > 0$ the pixel at $(x_i+1,y_i+1)$ is closest. The value of $d$ will change as we proceed along the line, let $\Delta d = d_{i+1} - d_i$ be the change in $d$ from one pixel to the next then we can calculate $d = d + \Delta d$ using

\begin{align*}
    \Delta d &= d_{i+1} - d_{i} \\
    &= f(x_i + 2, y_{i+1} + \tfrac{1}{2}) - f(x_i + 1, y_i + \tfrac{1}{2}) \\
    &= (x_i + 2) \Delta y - (y_{i+1} + \tfrac{1}{2}) \Delta x \\
    & \qquad + c \Delta x - (x_i + 1) \Delta y + (y_i + \tfrac{1}{2}) \Delta x - c \Delta x \\
    &= (x_i + 2) \Delta y - (x_i + 1) \Delta y - (y_{i+1} + \tfrac{1}{2}) \Delta x + (y_i + \tfrac{1}{2}) \Delta x
\end{align*}

The value of $\Delta d$ depends on which of the candidate nodes we have chosen. If $d \leq 0$ the midpoint is beneath the line and we choose the upper pixel $(x_i+1,y_i)$ so $y_{i+1} = y_i$ and

\begin{align*}
    \Delta d &= (x_i + 2) \Delta y - (x_i + 1) \Delta y - (y_i + \tfrac{1}{2}) \Delta x + (y_i + \tfrac{1}{2}) \Delta x \\
    &= x_i \Delta y + 2 \Delta y - x_i \Delta y - \Delta y - y_i \Delta x - \tfrac{1}{2} \Delta x + y_i \Delta x + \tfrac{1}{2} \Delta x \\
    &= \Delta y.
\end{align*}

Otherwise if $d > 0$ the midpoint is above the line and we choose the lower pixel $(x_i+1,y_i+1)$ so $y_{i+1} = y_i + 1$ and

\begin{align*}
    \Delta d &= (x_i + 2) \Delta y - (x_i + 1) \Delta y - (y_i + 1 + \tfrac{1}{2}) \Delta x + (y_i + \tfrac{1}{2}) \Delta x \\
    &= x_i \Delta y + 2 \Delta y - x_i \Delta y - \Delta y - y_i \Delta x - \Delta x - \tfrac{1}{2} \Delta x + y_i \Delta x + \tfrac{1}{2} \Delta x \\
    &= \Delta y - \Delta x.
\end{align*}

So we have way of choosing between the two candidate pixels. All we now need to do is determine the initial value of $d$. The first endpoint $(x_0,y_0)$ is plotted so the two candidate pixels at $(x_0+1,y_0)$ and $(x_0+1,y_0+1)$ so we calculate the initial value of $d$ using

\begin{align*}
    d &= f(x_0 + 1, y_0 + \tfrac{1}{2}) - f(x_0,y_0) \\
    &= (x_0 + 1) \Delta y - (y_0 + \tfrac{1}{2}) \Delta x + c \Delta x - x_0 \Delta y - y_0 \Delta x - c \Delta x \\
    &= x_0 \Delta y + \Delta y - y_0 \Delta x - \tfrac{1}{2} \Delta x - x_0 \Delta y - y_0 \Delta x \\
    &= \Delta y - \tfrac{1}{2} \Delta x.
\end{align*}

The expression for calculating includes a floating point number $\tfrac{1}{2}$. Floating point numbers have a much larger computational cost than integer numbers so it would be beneficial if we can only use integer numbers. Since we are only concerned with the sign of $d$ and not its value we can multiply the expressions for calculating the initial value of $d$ and $\Delta d$ by 2 to give

\begin{align*}
    d &= 2 \Delta y - \Delta x, \\
    \Delta d &= 
    \begin{cases}
        2 \Delta y, & d \leq 0, \\
        2 \Delta y - 2 \Delta x, & d > 0.
    \end{cases}
\end{align*}

The basic Bresenham's algorithm is written using [pseudocode](https://en.wikipedia.org/wiki/Pseudocode) in {prf:ref}`basic-bresenhams-algorithm`.

```{prf:algorithm} Bresenham's algorithm
:label: basic-bresenhams-algorithm

**Inputs** A raster array $R$, pixel co-ordinates of the endpoints of a straight line $(x_0, y_0)$ and $(x_1, y_1)$ and the colour of the line defined by the RBG triplet $colour$.

**Output** A raster array $R$.

- $x \gets x_0$ and $y \gets y_0$
- $\Delta x \gets x_1 - x_0$ and $\Delta y \gets y_1 - y_0$
- $d \gets 2 \Delta y - \Delta x$
- While true do 
    - $R(y,x) \gets colour$ (change the colour of the pixel $(x, y)$)
    - If $x = x_1$ and $y = y_1$ then
        - Terminate algorithm and return $R$
    - If $d \leq 0$ then
        - $d \gets d + 2 \Delta y$
    - Else
        - $y \gets y + 1$
        - $d \gets d + 2 \Delta y - 2 \Delta x$
    - $x \gets x + 1$
```

`````{prf:example} 
:class: seealso
:label: basic-bresenham-example

Use the basic Bresenham's algorithm to determine the co-ordinates of the pixels on the straight line joining the
two pixels with co-ordinates $(2, 2)$ and $(7, 5)$

````{dropdown} Solution
First we need to calculate $\Delta x$, $\Delta y$ and $d$

\begin{align*}
    \Delta x &= x_1 - x_0 = 7 - 2 = 5, \\
    \Delta y &= y_1 - y_0 = 5 - 2 = 3, \\
    d &= 2 \Delta y - \Delta x = 2(3) - 5 = 1.
\end{align*}

Stepping through the pixels:

\begin{align*}
    x &= 2, & y &= 2, & d &= 1 > 0 & \therefore d &= 1 + 2(3) - 2(5) = -3, \\
    x &= 3, & y &= 3, & d &= -3 \leq 0 & \therefore d &= -3 + 2(3) = 3, \\
    x &= 4, & y &= 3, & d &= 3 > 0 & \therefore d &= 3 + 2(3) - 2(5) = -1, \\
    x &= 5, & y &= 4, & d &= -1 < 0 & \therefore d &= -1 + 2(3) = 5, \\
    x &= 6, & y &= 4, & d &= 5 > 0 & \therefore d &= 5 + 2(3) - 2(5) = 1, \\
    x &= 7, & y &= 5.
\end{align*}

So the pixels at $(2, 2)$, $(3, 3)$, $(4, 3)$, $(5, 4)$, $(6, 4)$ and $(7, 5)$ are plotted.

```{figure} /images/bresenham_example_2.png
:figwidth: 300px
```

````
`````

## Modified Bresenham's algorithm

In deriving the algorithm presented in {prf:ref}`basic-bresenhams-algorithm` we made the assumption that the line was drawn from left-to-right and top-to-bottom. To modify the algorithm so that it can draw lines in any direction we repeat the derivation considering steps in the $x$ and $y$ directions separately. This results in the modified Bresenham's algorithm shown in {prf:ref}`bresenhams-algorithm`. Note that in the modified algorithm we introduce a new decision variable $e$ which is used tests to determine whether the $x$ and $y$ co-ordinates are incremented.

```{prf:algorithm} Modified Bresenham's algorithm
:label: bresenhams-algorithm

**Inputs** A raster array $R$, pixel co-ordinates of the endpoints of a straight line $(x, y)$ and $(x_1, y_1)$ and the colour of the line defined by the RBG triplet $colour$.

**Output** A raster array $R$.

- $x \gets x_0$ and $y \gets y_0$
- $\Delta x \gets |x_1 - x|$ and $\Delta y \gets |y_1 - y|$
- $d \gets \Delta x - \Delta y$
- $x_{step} \gets 1$ and $y_{step} \gets 1$
- If $x > x_1$ then
    - $x_{step} \gets -1$ 
- If $y > y_1$ then
    - $y_{step} \gets -1$  
- While true do
    - $R(y,x) \gets colour$
    - If $x = x_1$ and $y = y_1$ then
        - Terminate algorithm and return $R$
    - $e \gets 2d$
    - If $e \geq -\Delta y$ then
        - $x \gets x + x_{step}$ and $d \gets d - \Delta y$ 
    - If $e \leq \Delta x$ then
        - $y \gets y + y_{step}$ and $d \gets d + \Delta x$    
```

`````{prf:example} 
:label: modified-bresenham-example
:class: seealso

Use Bresenham’s algorithm for drawing lines in any direction to determine the co-ordinates of
the pixels on the lines joining the following points:

&emsp; (i) &emsp; $(0,0)$ and $(4,6)$;

````{dropdown} Solution

\begin{align*}
    \Delta x &= |4 - 0| = 4, \\
    \Delta y &= |6 - 0| = 6, \\
    x_{step} &= 1, \\
    y_{step} &= 1, \\
    d &= 4 - 6 = -2.
\end{align*}
        
Stepping through the algorithm:

\begin{align*}
    e &= -4, \\
    x &= 1, & e &\geq -\Delta y & \therefore x &= 0 + 1 = 1, & d &= -2 - 6 = -8, \\
    y &= 1, & e &\leq \Delta x & \therefore y &= 0 + 1 = 1, & d &= -8 + 4 = -4, \\ \\
    e &= -8, \\
    x &= 2, & e &< -\Delta y, \\
    y &= 2, & e &\leq \Delta x & \therefore y &= 1 + 1 = 2, & d &= -4 + 4 = 0, \\ \\
    e &= 0, \\
    x &= 2, & e &\geq -\Delta y & \therefore x &= 1 + 1 = 2, & d &= 0 - 6 = -6, \\
    y &= 3, & e &\leq \Delta x & \therefore y &= 2 + 1 = 3, & d &= -6 + 4 = -2, \\ \\
    e &= -4, \\
    x &= 3, & e &\geq -\Delta y & \therefore x &= 2 + 1 = 3, & d &= -2 - 6 = -8, \\
    y &= 4, & e &\leq \Delta x & \therefore y &= 3 + 1 = 4, & d &= -8 + 4 = -4, \\ \\
    e &= -8, \\
    x &= 4, & e &< -\Delta y, \\
    y &= 5, & e &\leq \Delta x & \therefore y &= 4 + 1 = 5, & d &= -4 + 4 = 0, \\ \\
    e &= 0, \\
    x &= 4, & e &\geq -\Delta y & \therefore x &= 3 + 1 = 4, & d &= 0 - 6 = -6, \\
    y &= 6, & e &\leq \Delta x & \therefore y &= 5 + 1 = 6, & d &= -6 + 4 = -2, \\ \\
    x &= 5, \\
    y &= 7.
\end{align*}

```{figure} /images/bresenham_example_3_1.png
:figwidth: 300px
```

````

&emsp; (ii) &emsp; $(6,1)$ and $(2,4)$.

````{dropdown} Solution

\begin{align*}
    \Delta x &= |2 - 6| = 4, \\
    \Delta y &= |4 - 1| = 3, \\
    x_{step} &= -1, \\
    y_{step} &= 1, \\
    d &= 4 - 3 = 1.
\end{align*}
        
Stepping through the algorithm:
\begin{align*}
    e &= 2, \\
    x &= 7, & e &\geq -\Delta y & \therefore x &= 6 - 1 = 5, & d &= 1 - 3 = -2, \\
    y &= 2, & e &\leq \Delta x & \therefore y &= 1 + 1 = 2, & d &= -2 + 4 = 2, \\ \\
    e &= 4, \\
    x &= 6, & e &\geq -\Delta y & \therefore x &= 5 - 1 = 4, & d &= 2 - 3 = -1, \\
    y &= 3, & e &\leq \Delta x & \therefore y &= 2 + 1 = 3, & d &= -1 + 4 = 3, \\ \\
    e &= 6, \\
    x &= 5, & e &\geq -\Delta y & \therefore x &= 4 - 1 = 3, & d &= 3 - 3 = 0, \\
    y &= 4, & e &> \Delta x \\ \\
    e &= 0, \\
    x &= 4, & e &\geq -\Delta y & \therefore x &= 3 - 1 = 2, \\
    y &= 4, & e &\leq \Delta x & \therefore y &= 3 + 1 = 4, \\ \\
    x &= 3, \\
    y &= 5.
\end{align*}

```{figure} /images/bresenham_example_3_2.png
:figwidth: 300px
```

````
`````